### GDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = 'NLP/Thesis'

import os
os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

Mounted at /content/drive


'/content/drive/MyDrive/NLP/Thesis'

###Constants

In [ ]:
# Important note: this was the incorrect implementation of few-shot learning, since it was actually implemented as an
# -- approach were the images came from random countries, so random labels
# It would have been better to call it In context learning
# -- so something like "Random Context Learning"
# I will keep the old variable names to show the approach nonetheless.

# Standardized few-shot messaging
FEWSHOT_SYSTEM_MESSAGE = "You are a helpful assistant for image geolocation. Learn from the example images and their locations."

FEWSHOT_PREAMBLE = (
    "I will show you several example room images with their known locations. "
    "Learn from these examples, then analyze a new query image.\n\n"
    "EXAMPLE IMAGES:\n"
)

FEWSHOT_QUERY_INTRO = "\n" + "="*50 + "\nQUERY IMAGE TO ANALYZE:\n"

Huggingface

In [ ]:
# %pip install --upgrade huggingface_hub
from huggingface_hub import login as hf_login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
hf_login(token = hf_token)

###Imports

In [ ]:
%pip install unidecode --quiet
%pip install torchao --quiet
# %pip install -U bitsandbytes --quiet

from unidecode import unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.9 MB/s eta 0:00:00


In [ ]:
# Standard library imports
import json
import math
import re
import gc
import time
import random
import base64
import pickle

import subprocess
import shutil

import xml.etree.ElementTree as ET
from difflib import SequenceMatcher

from io import BytesIO
from pathlib import Path

from typing import Any, Dict, List, Optional, Union, Tuple
from abc import ABC, abstractmethod
from collections import defaultdict
from functools import lru_cache

# Third-party shared imports
import requests
from PIL import Image
from tqdm import tqdm
import numpy as np
from sklearn.cluster import KMeans

#### Gemma

In [ ]:
from transformers import (
    Gemma3ForConditionalGeneration,
    pipeline,
)

#### Qwen

In [ ]:
# Install from source (as recommended on the page)
%pip install git+https://github.com/huggingface/transformers accelerate
%pip install qwen-vl-utils[decord]==0.0.8

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-jfqbjsah
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-jfqbjsah
  Resolved https://github.com/huggingface/transformers to commit a85d6b2a1ecae042cba82663080d4c9b42fceb72
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.5/534.5 kB 35.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11148051 sha256=cfc4449024cf8fad7b1f94cc5ff3752772851a89aff01b5c4ba483cff1125a3e
  Stored in directory: /tmp/pip-ephem-wheel-cache-5bvthpuy/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
 

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    AutoTokenizer,
)

####LLama

In [ ]:
%pip install ollama
import ollama

In [ ]:
# Install/upgrade required packages
%pip install transformers==4.49.0
# %pip install flash-attn --no-build-isolation
%pip install timm einops

from transformers import AutoProcessor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.0 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3


#### MiniCPM

In [ ]:
# Essential packages
%pip install transformers>=4.40.0
%pip install torch>=2.1.2 torchvision
%pip install pillow>=10.1.0
%pip install accelerate bitsandbytes

import transformers
import torch
from PIL import Image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.2 MB/s eta 0:00:00


####LLaVA

In [ ]:
%pip install transformers>=4.40.0
%pip install torch>=2.1.2
%pip install pillow>=10.1.0
%pip install accelerate
%pip install -U bitsandbytes

# %pip install flash-attn --no-build-isolation

# LLaVA may need these
%pip install sentencepiece  # For Vicuna tokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.6 MB/s eta 0:00:00


#### Optimization

In [ ]:
import torch

from transformers import (
    TorchAoConfig,
    BitsAndBytesConfig,
)

###Geolocation Classes

#### Base Class

In [ ]:
# Create a persistent session for HTTP requests
# Connection pooling dramatically speeds up repeated requests to the same hosts
_HTTP_SESSION = requests.Session()
_HTTP_SESSION.headers.update({'User-Agent': 'Mozilla/5.0'})

class BaseGeolocationTester(ABC):

    def __init__(self,
                 model_name: str,
                 device: str = "cuda",
                 mode: str = "auto",
                 hotel50k_image_index: Optional[Union[str, Dict[str, str]]] = None,
                 airbnb_image_index: Optional[Union[str, Dict[str, str]]] = None,
                 lazy_load_indices: bool = False):  # New parameter for lazy loading

        self.model_name = model_name
        self.device = device
        self.mode = mode

        # Optimization: Lazy loading option for large indices
        # Why: If indices are huge, loading them upfront wastes time if not used
        # Trade-off: Slightly slower first access, but faster startup
        if lazy_load_indices:
            self._hotel50k_index_source = hotel50k_image_index
            self._airbnb_index_source = airbnb_image_index
            self._hotel50k_index = None
            self._airbnb_index = None
        else:
            self.hotel50k_index = self._load_image_index(hotel50k_image_index, "Hotel50k")
            self.airbnb_index = self._load_image_index(airbnb_image_index, "Airbnb")

        # Model components
        self.pipe = None
        self.model = None
        self.processor = None

        print(f"Initializing {self.__class__.__name__} with {model_name} in {mode} mode")

        # Delegate initialization to subclass
        self._initialize_model()

    # Property accessors for lazy loading
    @property
    def hotel50k_index(self) -> Dict[str, str]:
        """Lazy-load hotel50k index if needed."""
        if self._hotel50k_index is None and hasattr(self, '_hotel50k_index_source'):
            self._hotel50k_index = self._load_image_index(
                self._hotel50k_index_source, "Hotel50k"
            )
        return self._hotel50k_index if self._hotel50k_index is not None else {}

    @hotel50k_index.setter
    def hotel50k_index(self, value):
        self._hotel50k_index = value

    @property
    def airbnb_index(self) -> Dict[str, str]:
        """Lazy-load airbnb index if needed."""
        if self._airbnb_index is None and hasattr(self, '_airbnb_index_source'):
            self._airbnb_index = self._load_image_index(
                self._airbnb_index_source, "Airbnb"
            )
        return self._airbnb_index if self._airbnb_index is not None else {}

    @airbnb_index.setter
    def airbnb_index(self, value):
        self._airbnb_index = value

    @abstractmethod
    def _initialize_model(self):
        """Subclasses must implement this to initialize their specific model."""
        pass

    @abstractmethod
    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:
        """Format messages for zero-shot inference."""
        pass

    @abstractmethod
    def _format_messages_fewshot(self, query_image_url: str,
                                 support_samples: List[Dict],
                                 prompt: str) -> Any:
        """Format messages for few-shot inference."""
        pass

    @abstractmethod
    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """Run inference given formatted input."""
        pass

    # Optimized Image Loading Method

    def _resize_image(self, image: Image.Image, max_size: int = 336) -> Image.Image:
        """Standardized resizing to prevent KV Cache stalls in multi-image inference."""
        if max(image.size) > max_size:
            ratio = max_size / max(image.size)
            new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
            return image.resize(new_size, Image.LANCZOS)
        return image

    def _load_image_index(self, index_source: Optional[Union[str, Dict]],
                          dataset_name: str) -> Dict[str, str]:
        """
        Load image index from file path or use provided dict.
        """
        if index_source is None:
            return {}

        if isinstance(index_source, dict):
            print(f"{dataset_name} index: {len(index_source)} mappings")
            return index_source

        index_path = Path(index_source)
        if not index_path.exists():
            print(f"Warning: {dataset_name} index not found")
            return {}

        try:
            if index_path.suffix == '.json':
                with open(index_path, 'r') as f:
                    index = json.load(f)
            elif index_path.suffix == '.pkl':
                with open(index_path, 'rb') as f:
                    index = pickle.load(f)
            else:
                print(f"Warning: Unsupported file type for {dataset_name}")
                return {}

            print(f"{dataset_name} index: {len(index)} mappings loaded")
            return index

        except Exception as e:
            print(f"Warning: Failed to load {dataset_name} index: {e}")
            return {}

    def _resolve_image_source(self, url: str, image_id=None, source=None,
                          return_metadata: bool = False) -> Union[str, Tuple[str, bool]]:
        """
        Resolve image source.

        Args:
            return_metadata: If True, returns (path, is_local). If False, returns just path.
        """
        # Quick check: local file?
        if not url.startswith(('http://', 'https://')):
            path = url
            is_local = Path(url).exists()
            return (path, is_local) if return_metadata else path

        # Check indices
        if image_id is not None:
            image_id_str = str(image_id)

            if source == 'hotel50k':
                local_path = self.hotel50k_index.get(image_id_str)
                if local_path and Path(local_path).exists():
                    return (local_path, True) if return_metadata else local_path
            elif source == 'airbnb':
                local_path = self.airbnb_index.get(image_id_str)
                if local_path and Path(local_path).exists():
                    return (local_path, True) if return_metadata else local_path
            else:
                local_path = self.hotel50k_index.get(image_id_str) or \
                            self.airbnb_index.get(image_id_str)
                if local_path and Path(local_path).exists():
                    return (local_path, True) if return_metadata else local_path

        return (url, False) if return_metadata else url

    def _load_image_from_url(self, url: str, image_id=None, source=None,
                         force_url: bool = False) -> Image.Image:
        """
        Load image from URL or local path.

        Args:
            force_url: If True, skip local index lookup and download directly.
                      Use this to avoid slow Google Drive FUSE access.
        """
        # Fast path: direct URL download (skips slow Drive access)
        if force_url or not url.startswith(('http://', 'https://')):
            if not url.startswith(('http://', 'https://')):
                # It's a local path, just open it
                return Image.open(url).convert('RGB')

            # Download from URL directly
            response = _HTTP_SESSION.get(url, timeout=15)
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert('RGB')

        # Standard path: try local index first (only if force_url=False)
        resolved_source, is_local = self._resolve_image_source(
            url, image_id, source, return_metadata=True
        )

        if is_local:
            return Image.open(resolved_source).convert('RGB')

        # Fallback to URL
        response = _HTTP_SESSION.get(resolved_source, timeout=15)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert('RGB')

    def _format_ground_truth_simple(self, metadata: Dict) -> str:
        """Format ground truth as readable text for few-shot examples."""
        return (f"Location: {metadata.get('city_name', 'unknown')}, "
                f"{metadata.get('state_admin1', 'unknown')}, "
                f"{metadata.get('country', 'unknown')}\n"
                f"Continent: {metadata.get('continent', 'unknown')}\n"
                f"Coordinates: {metadata.get('latitude', 0.0)}, "
                f"{metadata.get('longitude', 0.0)}")

    # Inference method

    def generate_prediction_zeroshot(self,
                                     image_url: str,
                                     prompt: str,
                                     image_id: Optional[Union[int, str]] = None,
                                     source: Optional[str] = None,
                                     max_new_tokens: int = 1024) -> str:
        """
        Zero-shot inference - same structure for all models.

        Note: Image loading happens in subclass _run_inference
        Why: Different models have different image processing pipelines
        """
        formatted_input = self._format_messages_zeroshot(image_url, prompt)
        response = self._run_inference(formatted_input, max_new_tokens)
        return response

    def generate_prediction_fewshot(self,
                                    query_image_url: str,
                                    query_image_id: Union[int, str],
                                    support_samples: List[Dict],
                                    prompt: str,
                                    query_source: Optional[str] = None,
                                    max_new_tokens: int = 1024,
                                    debug_first: bool = False) -> str:
        """
        Few-shot inference with optimized image preloading.

        Optimization: Removed image preloading from base class
        Why:
        1. Preloading doesn't help - images get cached on first access anyway
        2. Loading without using wastes time (LRU cache helps with actual reuse)
        3. Different models might process images differently
        4. Let subclass handle loading at inference time

        Result: Faster few-shot inference, cleaner code
        """
        # Just format and run inference
        # The subclass will load images when needed, benefiting from cache
        formatted_input = self._format_messages_fewshot(
            query_image_url, support_samples, prompt
        )

        response = self._run_inference(formatted_input, max_new_tokens)
        return response

    # XML Parsing Method

    def parse_xml_response(self, response: str) -> Optional[Dict[str, Any]]:
        """Parse XML response - same for all models."""
        if not response:
            return None

        code_match = re.search(
            r'```(?:xml)?\s*(<location>.*?</location>)\s*```',
            response, re.DOTALL | re.IGNORECASE
        )
        if code_match:
            result = self._parse_xml_string(code_match.group(1))
            if result:
                return result

        loc_match = re.search(
            r'(<location>.*?</location>)',
            response, re.DOTALL | re.IGNORECASE
        )
        if loc_match:
            result = self._parse_xml_string(loc_match.group(1))
            if result:
                return result

        return self._extract_from_fragments(response)

    def _parse_xml_string(self, xml_str: str) -> Optional[Dict[str, Any]]:
        """Parse complete XML string."""
        try:
            xml_str = xml_str.strip()
            xml_str = re.sub(r'&(?!(amp|lt|gt|quot|apos);)', r'&amp;', xml_str)
            root = ET.fromstring(xml_str)
            return self._extract_from_element(root)
        except ET.ParseError:
            return None

    def _extract_from_element(self, root: ET.Element) -> Dict[str, Any]:
        """Extract data from parsed XML element. Works for both zero-shot and few-shot."""
        def get_text(parent, tag):
            child = parent.find(tag)
            return child.text.strip() if child is not None and child.text else None

        def get_float(parent, tag):
            text = get_text(parent, tag)
            if text:
                try:
                    return float(text)
                except ValueError:
                    return None
            return None

        coords = root.find('coordinates')
        lat = get_float(coords, 'latitude') if coords is not None else None
        lon = get_float(coords, 'longitude') if coords is not None else None

        property_elem = root.find('property') or root.find('hotel')
        property_name = get_text(property_elem, 'name') if property_elem is not None else None
        property_type = get_text(property_elem, 'type') if property_elem is not None else None
        property_chain = get_text(property_elem, 'chain') if property_elem is not None else None

        return {
            # Core fields (zero-shot and few-shot)
            'continent': get_text(root, 'continent'),
            'country': get_text(root, 'country'),
            'region': get_text(root, 'region'),
            'city': get_text(root, 'city'),
            'latitude': lat,
            'longitude': lon,
            'confidence': get_text(root, 'confidence'),
            'property_name': property_name,
            'property_type': property_type,
            'property_chain': property_chain,
            'reasoning': get_text(root, 'reasoning'),
            # Few-shot specific fields (will be None for zero-shot responses)
            'example_used': get_text(root, 'example_used'),
            'example_reasoning': get_text(root, 'example_reasoning'),
        }



    def _extract_from_fragments(self, response: str) -> Optional[Dict[str, Any]]:
        """Extract from XML fragments when full parse fails. Works for both zero-shot and few-shot."""
        patterns = {
            # Core fields
            'continent': r'<continent>(.*?)</continent>',
            'country': r'<country>(.*?)</country>',
            'region': r'<region>(.*?)</region>',
            'city': r'<city>(.*?)</city>',
            'latitude': r'<latitude>(.*?)</latitude>',
            'longitude': r'<longitude>(.*?)</longitude>',
            'confidence': r'<confidence>(.*?)</confidence>',
            'property_name': r'<name>(.*?)</name>',
            'property_type': r'<type>(.*?)</type>',
            'property_chain': r'<chain>(.*?)</chain>',
            'reasoning': r'<reasoning>(.*?)</reasoning>',
            # Few-shot specific fields
            'example_used': r'<example_used>(.*?)</example_used>',
            'example_reasoning': r'<example_reasoning>(.*?)</example_reasoning>',
        }

        result = {}
        found_any = False

        for key, pattern in patterns.items():
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                value = match.group(1).strip()
                if key in ['latitude', 'longitude']:
                    try:
                        result[key] = float(value)
                        found_any = True
                    except ValueError:
                        result[key] = None
                else:
                    result[key] = value if value else None
                    if value:
                        found_any = True
            else:
                result[key] = None

        return result if found_any else None


    def parse_xml_response_attribution(self, response: str) -> Optional[Dict[str, Any]]:
        """
        Parse attribution-style XML with fallback to basic parsing.

        This method handles responses where the model lists key visual features
        that led to its location prediction. Each feature is in a separate
        key_feature_N tag with an explanation of why it's relevant.

        Example attribution XML structure:
        <location>
          <continent>North America</continent>
          <country>United States</country>
          <key_feature_1>Electrical outlets are Type A/B (North American standard)</key_feature_1>
          <key_feature_2>Window design with double-hung sash typical of US construction</key_feature_2>
          <reasoning>Overall synthesis of evidence points to US location</reasoning>
        </location>

        The method tries to parse this structure first, and if it fails or
        doesn't find the expected format, it falls back to the basic parser.
        This fallback ensures we always extract something useful even if
        the model didn't follow the attribution format perfectly.
        """
        if not response:
            return None

        # Try to find XML in code blocks first (common model behavior)
        # Models often wrap XML in markdown code blocks like ```xml ... ```
        for pattern in [r'```(?:xml)?\s*(<location>.*?</location>)\s*```',
                        r'(<location>.*?</location>)']:
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                result = self._parse_xml_string_attribution(match.group(1))
                if result:
                    return result

        # If attribution parsing failed, fall back to basic parser
        # This is defensive programming - even if the model didn't follow
        # the attribution format, we might still get a valid basic XML response
        print("Warning: Attribution structure not found, falling back to basic parser")
        return self.parse_xml_response(response)

    def _parse_xml_string_attribution(self, xml_str: str) -> Optional[Dict[str, Any]]:
        """
        Parse attribution XML structure and extract key features.

        This is the actual parsing logic for attribution-style XML.
        It extracts the basic location fields plus up to 5 key features
        that the model identified as important visual clues.
        """
        try:
            # Clean up the XML string to make it parseable
            xml_str = xml_str.strip()
            # Fix unescaped ampersands (common XML error)
            xml_str = re.sub(r'&(?!(amp|lt|gt|quot|apos);)', r'&amp;', xml_str)
            root = ET.fromstring(xml_str)

            def get_text(parent, tag):
                """Extract text from an XML tag, handling missing tags gracefully."""
                child = parent.find(tag)
                return child.text.strip() if child is not None and child.text else None

            def get_float(parent, tag):
                """Extract a float from an XML tag, with error handling."""
                text = get_text(parent, tag)
                if text:
                    try:
                        return float(text)
                    except ValueError:
                        return None
                return None

            # Extract basic location info (same as standard parser)
            coords = root.find('coordinates')
            lat = get_float(coords, 'latitude') if coords is not None else None
            lon = get_float(coords, 'longitude') if coords is not None else None

            # Extract key features (this is what makes attribution special)
            # We look for key_feature_1 through key_feature_5
            key_features = []
            for i in range(1, 6):
                feature = get_text(root, f'key_feature_{i}')
                if feature:
                    key_features.append(feature)

            # Return a dictionary with all extracted information
            # Note the key_features field which is unique to attribution parsing
            return {
                'continent': get_text(root, 'continent'),
                'country': get_text(root, 'country'),
                'region': get_text(root, 'region'),
                'city': get_text(root, 'city'),
                'latitude': lat,
                'longitude': lon,
                'confidence': get_text(root, 'confidence'),
                'key_features': key_features,  # List of identified features
                'reasoning': get_text(root, 'reasoning')
            }
        except ET.ParseError:
            # XML parsing failed - return None and let caller fall back
            return None

    def parse_xml_response_hierarchical(self, response: str) -> Optional[Dict[str, Any]]:
        """
        Parse hierarchical-style XML with fallback to basic parsing.

        This method handles responses where the model provides reasoning
        at each level of geographic specificity. Instead of one overall
        reasoning field, we get separate explanations for why the model
        chose each continent, country, region, and city.

        This format is particularly useful for understanding the model's
        decision-making process.
        """
        if not response:
            return None

        # Try to find and parse hierarchical XML
        for pattern in [r'```(?:xml)?\s*(<location>.*?</location>)\s*```',
                        r'(<location>.*?</location>)']:
            match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
            if match:
                result = self._parse_xml_string_hierarchical(match.group(1))
                if result:
                    return result

        # Fallback to basic parser if hierarchical structure not found
        print("Warning: Hierarchical structure not found, falling back to basic parser")
        return self.parse_xml_response(response)

    def _parse_xml_string_hierarchical(self, xml_str: str) -> Optional[Dict[str, Any]]:
        """
        Parse hierarchical XML with separate reasoning at each geographic level.

        This extracts location fields paired with their corresponding reasoning.
        The result is a flat dictionary where each location field has an
        associated reasoning field (continent + continent_reasoning, etc.).
        """
        try:
            xml_str = xml_str.strip()
            xml_str = re.sub(r'&(?!(amp|lt|gt|quot|apos);)', r'&amp;', xml_str)
            root = ET.fromstring(xml_str)

            def get_text(parent, tag):
                child = parent.find(tag)
                return child.text.strip() if child is not None and child.text else None

            def get_float(parent, tag):
                text = get_text(parent, tag)
                if text:
                    try:
                        return float(text)
                    except ValueError:
                        return None
                return None

            # Extract coordinates (same as other parsers)
            coords = root.find('coordinates')
            lat = get_float(coords, 'latitude') if coords is not None else None
            lon = get_float(coords, 'longitude') if coords is not None else None

            # Extract location fields with their paired reasoning
            # This creates a dictionary with both location and reasoning fields
            return {
                'continent': get_text(root, 'continent'),
                'continent_reasoning': get_text(root, 'continent_reasoning'),
                'country': get_text(root, 'country'),
                'country_reasoning': get_text(root, 'country_reasoning'),
                'region': get_text(root, 'region'),
                'region_reasoning': get_text(root, 'region_reasoning'),
                'city': get_text(root, 'city'),
                'city_reasoning': get_text(root, 'city_reasoning'),
                'latitude': lat,
                'longitude': lon,
                'confidence': get_text(root, 'confidence')
            }
        except ET.ParseError:
            return None

#### Gemma

In [ ]:
from transformers import pipeline, AutoProcessor, Gemma3ForConditionalGeneration

class GemmaGeolocationTester(BaseGeolocationTester):

    def _initialize_model(self):
        """Initialize Gemma model and/or pipeline with optimizations."""
        print("  Loading Gemma with optimizations...")

        if self.mode == "pipeline":
            self._init_pipeline()
        elif self.mode == "model":
            self._init_model()
        elif self.mode == "auto":
            self._init_pipeline()
            self._init_model()
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

    def _init_pipeline(self):
        """Initialize pipeline for batch processing."""
        print("Loading Gemma pipeline...")
        self.pipe = pipeline(
            "image-text-to-text",
            model=self.model_name,
            torch_dtype=torch.bfloat16,
            # Better than manual device placement for pipeline
            device_map="auto"
        )
        print("Pipeline ready")

    def _init_model(self):
        """Initialize model for direct access with speed optimizations."""
        print("Loading Gemma processor and model...")

        self.processor = AutoProcessor.from_pretrained(self.model_name)

        # Load with optimizations
        self.model = Gemma3ForConditionalGeneration.from_pretrained(
            self.model_name,
            torch_dtype=torch.bfloat16,
            # Added for memory efficiency
            low_cpu_mem_usage=True
        )

        # Manual device placement for better control
        self.model = self.model.to(self.device)
        self.model.eval()

        # Verify loading
        sample_param = next(self.model.parameters())
        if sample_param.is_meta:
            raise RuntimeError("Gemma model failed to load - still on meta device")

        print(f"Model ready on {sample_param.device}")

        # Report memory
        if torch.cuda.is_available():
            vram_used = torch.cuda.memory_allocated() / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"VRAM: {vram_used:.1f}GB / {vram_total:.1f}GB")

    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:
        """
        Format messages for Gemma zero-shot.

        Note: Gemma's pipeline/processor handles URL loading internally.
        We just pass the URL as-is in the message structure.
        """
        return [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are a helpful assistant for image geolocation."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "url": image_url},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

    def _format_messages_fewshot(self, query_image_url: str,
                             support_samples: List[Dict],
                             prompt: str) -> Any:
        """Format messages for Gemma few-shot with standardized preamble."""
        user_content = [
            {"type": "text", "text": FEWSHOT_PREAMBLE}
        ]

        # Add support examples
        for i, support in enumerate(support_samples, 1):
            user_content.append({"type": "image", "url": support['image_url']})
            ground_truth = support['metadata']
            location_text = f"\nExample {i} Location:\n{self._format_ground_truth_simple(ground_truth)}\n"
            user_content.append({"type": "text", "text": location_text})

        # Add query
        user_content.extend([
            {"type": "text", "text": FEWSHOT_QUERY_INTRO},
            {"type": "image", "url": query_image_url},
            {"type": "text", "text": f"\n{prompt}"}
        ])

        return [
            {
                "role": "system",
                "content": [{"type": "text", "text": FEWSHOT_SYSTEM_MESSAGE}]
            },
            {"role": "user", "content": user_content}
        ]

    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """
        Run inference using Gemma's pipeline or model.

        Gemma's processor handles image loading from URLs internally,
        so we don't need to manually load images with base class methods.
        The processor will benefit from HTTP session pooling in the base class
        if images are cached.
        """
        if self.pipe is not None:
            # Use pipeline
            output = self.pipe(text=formatted_input, max_new_tokens=max_new_tokens)
            return output[0]["generated_text"][-1]["content"].strip()

        elif self.model is not None and self.processor is not None:
            # Use model directly
            inputs = self.processor.apply_chat_template(
                formatted_input,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            )

            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            input_len = inputs["input_ids"].shape[-1]

            with torch.inference_mode():
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id
                )
                output_ids = output_ids[0][input_len:]

            response = self.processor.decode(output_ids, skip_special_tokens=True).strip()

            # Cleanup
            del inputs, output_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return response

        else:
            raise RuntimeError("No model or pipeline available")

    def generate_predictions_batch(self, image_urls: List[str],
                                   prompt: str,
                                   max_new_tokens: int = 1024) -> List[str]:
        """Batch processing using Gemma pipeline."""
        if self.pipe is None:
            raise RuntimeError("Batch processing requires pipeline mode")

        batch_messages = [
            self._format_messages_zeroshot(url, prompt)
            for url in image_urls
        ]

        outputs = self.pipe(
            text=batch_messages,
            max_new_tokens=max_new_tokens,
            batch_size=len(image_urls)
        )

        return [output[0]["generated_text"][-1]["content"].strip()
                for output in outputs]

#### Qwen

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

class QwenGeolocationTester(BaseGeolocationTester):

    def _initialize_model(self):
        """Initialize Qwen model with optimizations."""
        if self.mode == "pipeline":
            print("Warning: Qwen doesn't support pipeline mode, using model mode")
            self.mode = "model"

        self._init_model()

    def _init_model(self):
        """Initialize Qwen model and processor with memory optimizations."""
        print("  Loading Qwen processor and model...")

        # Load model with better device handling
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.model_name,
            # Explicit dtype instead of "auto"
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )

        # Manual device placement for better control
        self.model = self.model.to(self.device)
        self.model.eval()

        # Verify loading
        sample_param = next(self.model.parameters())
        if sample_param.is_meta:
            raise RuntimeError("Qwen model failed to load - still on meta device")

        # Reduce resolution to save memory
        min_pixels = 256 * 28 * 28  # ~200K pixels
        max_pixels = 512 * 28 * 28  # ~400K pixels (instead of default 1280*28*28)

        self.processor = AutoProcessor.from_pretrained(
            self.model_name,
            min_pixels=min_pixels,
            max_pixels=max_pixels
        )

        print(f"Model ready on {sample_param.device}")

        # Report memory
        if torch.cuda.is_available():
            vram_used = torch.cuda.memory_allocated() / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"VRAM: {vram_used:.1f}GB / {vram_total:.1f}GB")

    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:
        """
        Format messages for Qwen zero-shot.

        Qwen uses "image" key instead of "url".
        """
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_url},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

    def _format_messages_fewshot(self, query_image_url: str,
                             support_samples: List[Dict],
                             prompt: str) -> Any:
        """Format messages for Qwen few-shot with standardized preamble."""
        content = [
            {"type": "text", "text": FEWSHOT_PREAMBLE}
        ]

        # Add support examples
        for i, support in enumerate(support_samples, 1):
            content.append({"type": "image", "image": support['image_url']})
            ground_truth = support['metadata']
            location_text = f"\nExample {i} Location:\n{self._format_ground_truth_simple(ground_truth)}\n"
            content.append({"type": "text", "text": location_text})

        # Add query
        content.extend([
            {"type": "text", "text": FEWSHOT_QUERY_INTRO},
            {"type": "image", "image": query_image_url},
            {"type": "text", "text": f"\n{prompt}"}
        ])

        return [{"role": "user", "content": content}]

    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """
        Run inference using Qwen model.
        Qwen's process_vision_info handles image loading.
        """
        # Prepare text for Qwen processor
        text = self.processor.apply_chat_template(
            formatted_input,
            tokenize=False,
            add_generation_prompt=True
        )

        # Process images and text
        image_inputs, video_inputs = process_vision_info(formatted_input)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )

        inputs = inputs.to(self.device)
        input_len = inputs.input_ids.shape[1]

        # Context check
        if input_len > 32768:
            raise ValueError(f"Input length {input_len} exceeds Qwen context limit")

        # Generate
        with torch.inference_mode():
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )

            # Trim input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

        # Decode
        output_text = self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        # Cleanup
        del inputs, generated_ids, generated_ids_trimmed
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return output_text[0].strip()

####LLama

In [ ]:
class OllamaGeolocationTester(BaseGeolocationTester):
    """
    Ollama implementation adapted for optimized base class.
    """

    def _initialize_model(self):
        """Initialize Ollama client - no loading needed, Ollama manages the model."""
        print(f"  Initializing Ollama client for {self.model_name}...")

        # Ollama client is stateless - we just need to ensure the model is available
        try:
            # Check if model is available
            models_response = ollama.list()

            # Extract model names correctly
            model_names = [m.model for m in models_response.get('models', [])]

            if self.model_name not in model_names:
                print(f"Warning: {self.model_name} not found in Ollama.")
                print(f"Available models: {model_names}")
                print(f"Pulling {self.model_name}...")
                ollama.pull(self.model_name)
            else:
                print(f"Model {self.model_name} is available")

        except Exception as e:
            print(f"Error checking Ollama models: {e}")
            print("Attempting to use model anyway...")

        print("Ollama client ready")

    def _image_to_bytes(self, image: Image.Image) -> bytes:
        """Convert PIL Image to bytes for Ollama."""
        buffered = BytesIO()
        # Convert to RGB if needed
        if image.mode != 'RGB':
            image = image.convert('RGB')
        image.save(buffered, format="JPEG", quality=95)
        return buffered.getvalue()

    def _resize_image(self, image: Image.Image, max_size: int = 512) -> Image.Image:
        """
        Standardized resizing to reduce processing time and VRAM usage.
        """
        if max(image.size) > max_size:
            ratio = max_size / max(image.size)
            new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
            # Use Image.LANCZOS for high-quality downsampling
            return image.resize(new_size, Image.LANCZOS)
        return image

    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:
        """
        Format for zero-shot.
        Returns metadata for loading in _run_inference.
        """
        return {
            'image_url': image_url,
            'prompt': prompt,
            'mode': 'zeroshot',
            'image_id': None,
            'source': None
        }

    def _format_messages_fewshot(self, query_image_url: str,
                                 support_samples: List[Dict],
                                 prompt: str) -> Any:
        """Format for few-shot."""
        return {
            'query_image_url': query_image_url,
            'support_samples': support_samples,
            'prompt': prompt,
            'mode': 'fewshot',
            'query_image_id': None,
            'query_source': None
        }

    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """Run inference - delegates to specific methods."""
        mode = formatted_input['mode']

        if mode == 'zeroshot':
            return self._run_zeroshot_inference(formatted_input, max_new_tokens)
        else:
            return self._run_fewshot_inference(formatted_input, max_new_tokens)

    def _run_zeroshot_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """
        Zero-shot inference with Ollama.
        """
        image_url = formatted_input['image_url']
        prompt = formatted_input['prompt']
        image_id = formatted_input.get('image_id')
        source = formatted_input.get('source')

        # Load image using optimized base class method
        # This benefits from HTTP session pooling and caching
        image = self._load_image_from_url(image_url, image_id, source)

        # Convert to bytes for Ollama
        image_bytes = self._image_to_bytes(image)

        # Make Ollama chat API call
        try:
            response = ollama.chat(
                model=self.model_name,
                messages=[
                    {
                        'role': 'user',
                        'content': prompt,
                        'images': [image_bytes]
                    }
                ],
                options={
                    'num_predict': max_new_tokens,
                    'temperature': 0.0,  # Deterministic
                }
            )

            return response['message']['content'].strip()

        except Exception as e:
            raise Exception(f"Ollama inference failed: {e}")

    def _run_fewshot_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        query_image_url = formatted_input['query_image_url']
        support_samples = formatted_input['support_samples']
        prompt = formatted_input['prompt']

        messages = [
            {'role': 'system', 'content': FEWSHOT_SYSTEM_MESSAGE},
            {'role': 'user', 'content': FEWSHOT_PREAMBLE},
            {'role': 'assistant', 'content': 'I understand. Please show me the examples.'}
        ]

        for i, support in enumerate(support_samples, 1):
            try:
                # force_url = True to skip slow Drive lookup
                img = self._load_image_from_url(support['image_url'], force_url=True)
                img = self._resize_image(img, max_size=336)
                img_bytes = self._image_to_bytes(img)

                messages.append({'role': 'user', 'content': f'Example {i}:', 'images': [img_bytes]})
                messages.append({
                    'role': 'assistant',
                    'content': f"Example {i} Location:\n{self._format_ground_truth_simple(support['metadata'])}"
                })
            except Exception as e:
                print(f"  Warning: Skipping support image {i}: {e}")

        # Query image
        try:

            query_image = self._load_image_from_url(query_image_url, force_url=True)
            query_image = self._resize_image(query_image, max_size=512)

            messages.append({
                'role': 'user',
                'content': FEWSHOT_QUERY_INTRO + prompt,
                'images': [self._image_to_bytes(query_image)]
            })

            response = ollama.chat(
                model=self.model_name,
                messages=messages,
                options={'num_predict': max_new_tokens, 'temperature': 0.0}
            )

            return response['message']['content'].strip()

        except Exception as e:
            raise Exception(f"Ollama few-shot failure: {e}")

    def generate_prediction_zeroshot(self,
                                     image_url: str,
                                     prompt: str,
                                     image_id: Optional[Union[int, str]] = None,
                                     source: Optional[str] = None,
                                     max_new_tokens: int = 1024) -> str:

        """Override to pass metadata."""
        formatted_input = self._format_messages_zeroshot(image_url, prompt)
        formatted_input['image_id'] = image_id
        formatted_input['source'] = source
        return self._run_inference(formatted_input, max_new_tokens)

    def generate_prediction_fewshot(self,
                                    query_image_url: str,
                                    query_image_id: Union[int, str],
                                    support_samples: List[Dict],
                                    prompt: str,
                                    query_source: Optional[str] = None,
                                    max_new_tokens: int = 1024,
                                    debug_first: bool = False) -> str:

        """Override to pass metadata."""
        formatted_input = self._format_messages_fewshot(
            query_image_url, support_samples, prompt
        )

        formatted_input['query_image_id'] = query_image_id
        formatted_input['query_source'] = query_source

        return self._run_inference(formatted_input, max_new_tokens)

####MiniCPM

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
from typing import Any, Dict, List, Optional, Union
from PIL import Image

class MiniCPMVGeolocationTesterOptimized(BaseGeolocationTester):

    def _initialize_model(self):
        print("Loading MiniCPM-V-2.6 with speed optimizations...")

        model_name_to_load = self.model_name.replace('-int4', '')

        # Pin revision to avoid repeated downloads
        revision = "main"  # or specific commit hash

        self.model = AutoModel.from_pretrained(
            model_name_to_load,
            trust_remote_code=True,
            attn_implementation='sdpa',
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            revision=revision
        )

        self.model = self.model.to(self.device)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_to_load,
            trust_remote_code=True,
            revision=revision
        )
        # Verify model is properly loaded
        print("Verifying model device placement...")

        # Check a sample parameter to ensure it's on GPU and not meta
        sample_param = next(self.model.parameters())
        if sample_param.is_meta:
            raise RuntimeError(
                "Model parameters are still on meta device."
                "This indicates the model wasn't properly loaded. "
                "Try restarting your kernel and loading again."
            )

        if sample_param.device.type != 'cuda':
            print(f"Warning: Model is on {sample_param.device}, not CUDA!")
        else:
            print(f"Model successfully loaded on {sample_param.device}")

        # Report memory usage
        if torch.cuda.is_available():
            vram_used = torch.cuda.memory_allocated() / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"Memory usage: {vram_used:.1f}GB / {vram_total:.1f}GB VRAM")

            # Sanity check: BF16 model should use ~15-16GB
            if vram_used < 10:
                print("Warning: VRAM usage is lower than expected for full BF16 model!")
                print("This suggests the model might still be quantized.")
                print("Expected: ~15-16GB for full model")
                print(f"Actual: {vram_used:.1f}GB")

    def _resize_image(self, image: Image.Image, max_size: int = 512) -> Image.Image:
        """Resize image to reduce processing time."""
        if max(image.size) > max_size:
            ratio = max_size / max(image.size)
            new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
            return image.resize(new_size, Image.LANCZOS)
        return image


    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:

        return {
            'image_url': image_url,
            'prompt': prompt,
            'mode': 'zeroshot',
            'image_id': None,  # Will be set if provided
            'source': None     # Will be set if provided
        }

    def _format_messages_fewshot(self, query_image_url: str,
                                 support_samples: List[Dict],
                                 prompt: str) -> Any:
        return {
            'query_image_url': query_image_url,
            'support_samples': support_samples,
            'prompt': prompt,
            'mode': 'fewshot',
            'query_image_id': None,  # Will be set if provided
            'query_source': None     # Will be set if provided
        }

    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        mode = formatted_input['mode']
        msgs = []

        if mode == 'zeroshot':
            image = self._load_image_from_url(
                formatted_input['image_url'],
                formatted_input.get('image_id'),
                formatted_input.get('source')
            )
            msgs = [{'role': 'user', 'content': [image, formatted_input['prompt']]}]
        else:
            # Few-shot: force_url = True to skip slow Drive
            msgs.append({'role': 'user', 'content': [FEWSHOT_PREAMBLE]})
            msgs.append({'role': 'assistant', 'content': ['I understand. Please show me the examples.']})

            for i, support in enumerate(formatted_input['support_samples'], 1):

                img = self._load_image_from_url(support['image_url'], force_url=True)
                img = self._resize_image(img, max_size=336)
                msgs.append({'role': 'user', 'content': [img, f"Example {i}:"]})
                msgs.append({'role': 'assistant', 'content': [self._format_ground_truth_simple(support['metadata'])]})

            query_img = self._load_image_from_url(formatted_input['query_image_url'], force_url=True)
            msgs.append({'role': 'user', 'content': [query_img, FEWSHOT_QUERY_INTRO + formatted_input['prompt']]})

        with torch.inference_mode():
            response = self.model.chat(
                image=None, msgs=msgs,
                tokenizer=self.tokenizer,
                sampling=False,
                max_new_tokens=max_new_tokens
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        return response.strip()

    def generate_prediction_zeroshot(self,
                                     image_url: str,
                                     prompt: str,
                                     image_id: Optional[Union[int, str]] = None,
                                     source: Optional[str] = None,
                                     max_new_tokens: int = 1024) -> str:
        """
        Override to pass image_id and source to _run_inference.
        """
        formatted_input = self._format_messages_zeroshot(image_url, prompt)
        formatted_input['image_id'] = image_id
        formatted_input['source'] = source
        response = self._run_inference(formatted_input, max_new_tokens)
        return response

    def generate_prediction_fewshot(self,
                                    query_image_url: str,
                                    query_image_id: Union[int, str],
                                    support_samples: List[Dict],
                                    prompt: str,
                                    query_source: Optional[str] = None,
                                    max_new_tokens: int = 1024,
                                    debug_first: bool = False) -> str:
        """
        Override to pass query metadata to _run_inference.
        """
        formatted_input = self._format_messages_fewshot(
            query_image_url, support_samples, prompt
        )

        formatted_input['query_image_id'] = query_image_id
        formatted_input['query_source'] = query_source

        response = self._run_inference(formatted_input, max_new_tokens)
        return response

####LLaVA

In [ ]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
import torch

class LLaVAGeolocationTester(BaseGeolocationTester):

    def _initialize_model(self):
        """Initialize with 8-bit quantization."""
        print("Loading LLaVA-NeXT (4-bit quantized)...")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        # Check Flash Attention
        try:
            import flash_attn
            attn_impl = 'flash_attention_2'
            print("  Using Flash Attention 2")
        except ImportError:
            attn_impl = 'sdpa'
            print("  Using SDPA")

        # Load processor
        self.processor = LlavaNextProcessor.from_pretrained(self.model_name)

        # Load 4-bit model
        print("Loading model in 4-bit")
        self.model = LlavaNextForConditionalGeneration.from_pretrained(
            self.model_name,
            quantization_config=quantization_config,
            device_map="auto",  # Required for quantization
            _attn_implementation=attn_impl,
        )

        # Gradient checkpointing for extra memory savings
        self.model.gradient_checkpointing_enable()
        print("4-bit quantization + gradient checkpointing enabled")

        self.model.eval()

        # Report memory
        if torch.cuda.is_available():
            vram = torch.cuda.memory_allocated() / 1e9
            vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"VRAM: {vram:.1f}GB / {vram_total:.1f}GB (saved ~7GB vs BF16)")

    def _format_messages_zeroshot(self, image_url: str, prompt: str) -> Any:
        return {
            'image_url': image_url, 'prompt': prompt, 'mode': 'zeroshot',
            'image_id': None, 'source': None
        }

    def _format_messages_fewshot(self, query_image_url: str,
                                 support_samples: List[Dict], prompt: str) -> Any:
        return {
            'query_image_url': query_image_url, 'support_samples': support_samples,
            'prompt': prompt, 'mode': 'fewshot', 'query_image_id': None, 'query_source': None
        }

    def _run_inference(self, formatted_input: Any, max_new_tokens: int) -> str:
        """Run inference with aggressive memory management."""
        # Clear cache before
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        try:
            if formatted_input['mode'] == 'zeroshot':
                return self._run_zeroshot(formatted_input, max_new_tokens)
            else:
                return self._run_fewshot(formatted_input, max_new_tokens)
        finally:
            # Clear cache after
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    def _run_zeroshot(self, formatted_input: Any, max_new_tokens: int) -> str:
        """Zero-shot inference."""
        image = self._load_image_from_url(
            formatted_input['image_url'],
            image_id=formatted_input.get('image_id'),
            source=formatted_input.get('source')
        )

        conversation = [{
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": formatted_input['prompt']}]
        }]

        prompt_text = self.processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = self.processor(images=image, text=prompt_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.processor.tokenizer.pad_token_id
            )

        generated_ids = output_ids[0][inputs['input_ids'].shape[1]:]
        response = self.processor.decode(generated_ids, skip_special_tokens=True)

        del inputs, output_ids, generated_ids
        return response.strip()

    def _run_fewshot(self, formatted_input: Any, max_new_tokens: int) -> str:
        support_samples = formatted_input['support_samples']
        conversation = []
        images = []

        conversation.append({
            "role": "user",
            "content": [{"type": "text", "text": FEWSHOT_PREAMBLE}]
        })
        conversation.append({
            "role": "assistant",
            "content": [{"type": "text", "text": "I understand. Please show me the example images with their locations."}]
        })

        # Now add support examples
        for i, support in enumerate(support_samples, 1):
            img = self._load_image_from_url(support['image_url'], force_url=True)
            images.append(self._resize_image(img, max_size=336))

            conversation.append({
                "role": "user",
                "content": [{"type": "image"}, {"type": "text", "text": f"Example {i}:"}]
            })
            conversation.append({
                "role": "assistant",
                "content": [{"type": "text", "text": f"Example {i} Location:\n{self._format_ground_truth_simple(support['metadata'])}"}]
            })

        # Query image
        query_image = self._load_image_from_url(formatted_input['query_image_url'], force_url=True)
        images.append(self._resize_image(query_image, max_size=512))
        conversation.append({
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": FEWSHOT_QUERY_INTRO + formatted_input['prompt']}]
        })

        prompt_text = self.processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = self.processor(images=images, text=prompt_text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.processor.tokenizer.pad_token_id
        )

        response = self.processor.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        del inputs, output_ids, images
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        gc.collect()
        return response.strip()

    def generate_prediction_zeroshot(self, image_url: str, prompt: str,
                                     image_id: Optional[Union[int, str]] = None,
                                     source: Optional[str] = None,
                                     max_new_tokens: int = 1024) -> str:

        formatted_input = self._format_messages_zeroshot(image_url, prompt)
        formatted_input['image_id'] = image_id
        formatted_input['source'] = source
        return self._run_inference(formatted_input, max_new_tokens)

    def generate_prediction_fewshot(self, query_image_url: str, query_image_id: Union[int, str],
                                    support_samples: List[Dict], prompt: str,
                                    query_source: Optional[str] = None,
                                    max_new_tokens: int = 1024,
                                    debug_first: bool = False) -> str:

        formatted_input = self._format_messages_fewshot(query_image_url, support_samples, prompt)
        formatted_input['query_image_id'] = query_image_id
        formatted_input['query_source'] = query_source
        return self._run_inference(formatted_input, max_new_tokens)


### Location Evaluator

In [ ]:
class LocationEvaluator:
    def __init__(self, thresholds: Optional[Dict[str, float]] = None):

        self.thresholds = thresholds or {
            'continent': 0.80,
            'country': 0.85,
            'region': 0.87,
            'city': 0.87
        }

    @staticmethod
    def normalize_text(text: Optional[str]) -> str:
        """Normalizes text for comparison."""
        if text is None or not isinstance(text, str):
            return "unknown"

        text = text.strip()
        if not text:
            return "unknown"

        try:
            text = unidecode(text)
        except:
            pass

        text = text.lower()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[.,\'"`]', '', text)

        # Normalize abbreviations
        text = re.sub(r'\bst\b', 'saint', text)
        text = re.sub(r'\bst\.', 'saint', text)
        text = re.sub(r'\bmt\b', 'mount', text)

        return text.strip() or "unknown"

    @staticmethod
    def calculate_similarity(str1: str, str2: str) -> float:
        """Calculates string similarity using SequenceMatcher."""
        return SequenceMatcher(None, str1, str2).ratio()

    def compare_locations(self, predicted: Optional[str],
                         ground_truth: Optional[str],
                         field_type: str) -> Dict[str, Any]:
        """
        Compares two location names with fuzzy matching.
        """
        pred_norm = self.normalize_text(predicted)
        truth_norm = self.normalize_text(ground_truth)

        threshold = self.thresholds.get(field_type, 0.85)

        result = {
            'match': False,
            'match_type': 'no_match',
            'similarity': 0.0,
            'predicted': predicted,
            'ground_truth': ground_truth,
            'predicted_normalized': pred_norm,
            'ground_truth_normalized': truth_norm
        }

        if pred_norm == "unknown" or truth_norm == "unknown":
            result['match_type'] = 'missing_data'
            return result

        if pred_norm == truth_norm:
            result['match'] = True
            result['match_type'] = 'exact'
            result['similarity'] = 1.0
            return result

        similarity = self.calculate_similarity(pred_norm, truth_norm)
        result['similarity'] = similarity

        if similarity >= threshold:
            result['match'] = True
            result['match_type'] = 'fuzzy'
            return result

        return result

    @staticmethod
    def calculate_distance(lat1: float, lon1: float,
                          lat2: float, lon2: float) -> float:
        """
        Calculates great circle distance using Haversine formula.
        Returns distance in kilometers.
        """
        lat1_rad = math.radians(lat1)
        lon1_rad = math.radians(lon1)
        lat2_rad = math.radians(lat2)
        lon2_rad = math.radians(lon2)

        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad

        a = (math.sin(dlat/2)**2 +
             math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2)
        c = 2 * math.asin(math.sqrt(a))

        # Earth radius in km
        return 6371 * c

    def evaluate_prediction(self, predicted: Dict[str, Any],
                      ground_truth: Dict[str, Any]) -> Dict[str, Any]:
        """
        Complete evaluation of a prediction against ground truth.
        """
        evaluation = {
            'extraction_successful': predicted is not None,
            'field_evaluations': {},
            'coordinate_evaluation': None,
            'overall_metrics': {}
        }

        field_mapping = {
            'continent': 'continent',
            'country': 'country',
            'region': 'state_admin1',
            'city': 'city_name'
        }

        if not predicted:
            evaluation['error'] = 'Failed to extract prediction'
            return evaluation

        # Evaluate text fields
        for field in ['continent', 'country', 'region', 'city']:
            pred_value = predicted.get(field)
            gt_field = field_mapping.get(field, field)
            gt_value = ground_truth.get(gt_field)

            comparison = self.compare_locations(pred_value, gt_value, field)
            evaluation['field_evaluations'][field] = comparison

        # Evaluate coordinates
        pred_lat = predicted.get('latitude')
        pred_lon = predicted.get('longitude')
        gt_lat = ground_truth.get('latitude')
        gt_lon = ground_truth.get('longitude')

        if all(v is not None for v in [pred_lat, pred_lon, gt_lat, gt_lon]):
            if -90 <= pred_lat <= 90 and -180 <= pred_lon <= 180:
                distance = self.calculate_distance(
                    pred_lat, pred_lon, gt_lat, gt_lon
                )

                evaluation['coordinate_evaluation'] = {
                    'distance_km': distance,
                    'within_1km': distance <= 1,
                    'within_10km': distance <= 10,
                    'within_100km': distance <= 100,
                    'within_1000km': distance <= 1000,
                    'predicted_coords': {'lat': pred_lat, 'lon': pred_lon},
                    'ground_truth_coords': {'lat': gt_lat, 'lon': gt_lon}
                }

        # Calculate overall metrics
        correct_fields = sum(
            1 for f in evaluation['field_evaluations'].values()
            if f['match']
        )
        evaluation['overall_metrics']['text_fields_correct'] = correct_fields
        evaluation['overall_metrics']['text_fields_total'] = 4

        return evaluation

### Evaluation functions

In [ ]:
def calculate_statistics(results: List[Dict]) -> Dict[str, Any]:
    total = len(results)
    successful = sum(
        1 for r in results
        if r.get('evaluation', {}).get('extraction_successful', False)
    )

    field_correct = {'continent': 0, 'country': 0, 'region': 0, 'city': 0}
    for result in results:
        field_evals = result.get('evaluation', {}).get('field_evaluations', {})
        for field, comparison in field_evals.items():
            if comparison.get('match', False):
                field_correct[field] += 1

    distances = []
    within_thresholds = {1: 0, 10: 0, 100: 0, 1000: 0}

    for result in results:
        coord_eval = result.get('evaluation', {}).get('coordinate_evaluation')
        if coord_eval and coord_eval.get('distance_km') is not None:
            dist = coord_eval['distance_km']
            distances.append(dist)
            for threshold in within_thresholds:
                if dist <= threshold:
                    within_thresholds[threshold] += 1

    stats = {
        'total_samples': total,
        'successful_extractions': successful,
        'extraction_success_rate': successful / total if total > 0 else 0,
        'continent_accuracy': field_correct['continent'] / total if total > 0 else 0,
        'country_accuracy': field_correct['country'] / total if total > 0 else 0,
        'region_accuracy': field_correct['region'] / total if total > 0 else 0,
        'city_accuracy': field_correct['city'] / total if total > 0 else 0,
    }

    if distances:
        stats['avg_distance_km'] = sum(distances) / len(distances)
        stats['median_distance_km'] = sorted(distances)[len(distances) // 2]
        stats['accuracy_1km'] = within_thresholds[1] / total
        stats['accuracy_10km'] = within_thresholds[10] / total
        stats['accuracy_100km'] = within_thresholds[100] / total
        stats['accuracy_1000km'] = within_thresholds[1000] / total

    return stats

def print_statistics_summary(stats: Dict, n_support: int):

    print(f"Evaluation Summary ({n_support}-Shot)")

    print(f"\nOutput Quality:")
    print(f"Successful extractions: {stats['extraction_success_rate']:.1%}")

    print(f"\nGeographic Accuracy:")
    print(f"Continent: {stats['continent_accuracy']:.1%}")
    print(f"Country:   {stats['country_accuracy']:.1%}")
    print(f"Region:    {stats['region_accuracy']:.1%}")
    print(f"City:      {stats['city_accuracy']:.1%}")

    if stats.get('avg_distance_km') is not None:
        print(f"\nCoordinate Accuracy:")
        print(f"Average distance:  {stats['avg_distance_km']:.1f} km")
        print(f"Median distance:   {stats['median_distance_km']:.1f} km")
        print(f"Within 1 km:       {stats['accuracy_1km']:.1%}")
        print(f"Within 10 km:      {stats['accuracy_10km']:.1%}")
        print(f"Within 100 km:     {stats['accuracy_100km']:.1%}")
        print(f"Within 1000 km:    {stats['accuracy_1000km']:.1%}")

### Run test functions

#### Zero shot

In [ ]:
def run_zeroshot_test(tester: BaseGeolocationTester,
                      dataset_path: str,
                      num_samples: int = 100,
                      output_dir: str = "./experiments-results",
                      prompt_style: str = "simple",
                      max_new_tokens: int = 1024,
                      batch_size: int = 4):

    """Run zero-shot test with prompt style selection."""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    print(f"Loading dataset from {dataset_path}...")
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)

    if num_samples < len(dataset):
        import random
        dataset = random.sample(dataset, num_samples)
        print(f"Number of samples smaller than the dataset size, {len(dataset)}, Sampled {num_samples} entries")

    evaluator = LocationEvaluator()

    # Select prompt and parser based on style
    prompt_map = {
        "simple": xml_prompt_simple,
        "detailed": xml_prompt_detailed,
        "analytical": xml_prompt_analytical,
        "attribution": xml_prompt_attribution,
        "hierarchical": xml_prompt_hierarchical
    }

    parser_map = {
        "simple": tester.parse_xml_response,
        "detailed": tester.parse_xml_response,
        "analytical": tester.parse_xml_response,
        "attribution": tester.parse_xml_response_attribution,
        "hierarchical": tester.parse_xml_response_hierarchical
    }

    if prompt_style not in prompt_map:
        raise ValueError(f"Unknown prompt_style: {prompt_style}")

    prompt = prompt_map[prompt_style]
    parser = parser_map[prompt_style]

    results = []
    print(f"\nRunning zero-shot inference with '{prompt_style}' prompt on {len(dataset)} samples...")

    # Batch or individual processing
    if hasattr(tester, 'generate_predictions_batch') and tester.pipe is not None:
        print(f"Using batch processing (batch_size={batch_size})...")

        if tester.hotel50k_index or tester.airbnb_index:
            print("Resolving sources with fallback...")
            for sample in tqdm(dataset, desc="Resolving"):
                sample['resolved_source'] = tester._resolve_image_source(
                    url=sample['image_url'],
                    image_id=sample['image_id'],
                    source=sample.get('source') or sample.get('metadata', {}).get('original_source'),
                    return_metadata=False  # Just get path string
                )
        else:
            for sample in dataset:
                sample['resolved_source'] = sample['image_url']

        for batch_start in tqdm(range(0, len(dataset), batch_size), desc="Batches"):
            batch_end = min(batch_start + batch_size, len(dataset))
            batch_samples = dataset[batch_start:batch_end]
            batch_sources = [s['resolved_source'] for s in batch_samples]

            try:
                batch_responses = tester.generate_predictions_batch(
                    batch_sources, prompt, max_new_tokens
                )

                for idx, (sample, response) in enumerate(zip(batch_samples, batch_responses)):
                    predicted = parser(response)  # Use selected parser
                    evaluation = evaluator.evaluate_prediction(predicted, sample['metadata'])

                    results.append({
                        'sample_id': batch_start + idx,
                        'image_id': sample['image_id'],
                        'image_url': sample['image_url'],
                        'source': sample.get('source', 'unknown'),
                        'ground_truth': sample['metadata'],
                        'raw_response': response,
                        'predicted': predicted,
                        'evaluation': evaluation
                    })

            except Exception as e:
                print(f"\nError at batch {batch_start}: {e}")
                # Add error records for each sample in failed batch
                for idx in range(len(batch_samples)):
                    results.append({
                        'sample_id': batch_start + idx,
                        'image_id': batch_samples[idx]['image_id'],
                        'error': str(e)
                    })

    else:
        print("Using individual processing...")
        for idx, sample in enumerate(tqdm(dataset, desc="Processing")):
            try:
                response = tester.generate_prediction_zeroshot(
                    image_url=sample['image_url'],
                    prompt=prompt,
                    image_id=sample['image_id'],
                    source=sample.get('source') or sample.get('metadata', {}).get('original_source'),
                    max_new_tokens=max_new_tokens
                )

                predicted = parser(response)  # Use selected parser
                evaluation = evaluator.evaluate_prediction(predicted, sample['metadata'])

                results.append({
                    'sample_id': idx,
                    'image_id': sample['image_id'],
                    'image_url': sample['image_url'],
                    'source': sample.get('source', 'unknown'),
                    'ground_truth': sample['metadata'],
                    'raw_response': response,
                    'predicted': predicted,
                    'evaluation': evaluation
                })

            except Exception as e:
                print(f"\nError at sample {idx}: {e}")
                results.append({
                    'sample_id': idx,
                    'image_id': sample['image_id'],
                    'error': str(e)
                })

    # Save results
    stats = calculate_statistics(results)
    timestamp = time.strftime("%Y%m%d_%H%M%S")

    model_name_short = tester.model_name.split('/')[-1]
    final_output = output_path / f"{model_name_short}_zeroshot_{prompt_style}_{timestamp}.json"

    output_data = {
        'metadata': {
            'model': tester.model_name,
            'model_class': tester.__class__.__name__,
            'num_samples': len(dataset),
            'mode': 'zero-shot',
            'prompt_style': prompt_style,
            'timestamp': timestamp
        },
        'statistics': stats,
        'results': results
    }

    with open(final_output, 'w') as f:
        json.dump(output_data, f, indent=2)

    print(f"\nResults saved to: {final_output}")
    print_statistics_summary(stats, 0)

    return output_data

#### Few shot

In [ ]:
def run_fewshot_test(tester: BaseGeolocationTester,
                     dataset_path: str,
                     output_dir: str = "./experiments-results",
                     max_new_tokens: int = 1024,
                     ):
    """
    Run few-shot test with standardized prompt for example utilization tracking.
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    print(f"Loading few-shot dataset from {dataset_path}...")
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)

    n_support = dataset[0]['n_support'] if dataset else 0
    print(f"Loaded {len(dataset)} samples with {n_support} support examples each")

    evaluator = LocationEvaluator()

    # Use the standardized few-shot prompt
    prompt = xml_prompt_fewshot

    results = []
    example_usage_stats = {'used': 0, 'not_used': 0, 'unclear': 0}

    print(f"\nRunning {n_support}-shot inference...")
    print(f"Model: {tester.model_name}")

    for idx, sample in enumerate(tqdm(dataset, desc=f"{n_support}-shot")):
        # Clear CUDA cache periodically to prevent OOM
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        try:
            query_image_url = sample['query']['image_url']
            query_image_id = sample['query']['image_id']
            query_source = sample['query'].get('source') or \
                          sample['query']['metadata'].get('original_source')

            response = tester.generate_prediction_fewshot(
                query_image_url=query_image_url,
                query_image_id=query_image_id,
                support_samples=sample['support_samples'],
                prompt=prompt,
                query_source=query_source,
                max_new_tokens=max_new_tokens,
                debug_first=(idx == 0)
            )

            # Parse response (uses updated parser that handles few-shot fields)
            predicted = tester.parse_xml_response(response)
            evaluation = evaluator.evaluate_prediction(
                predicted,
                sample['query']['metadata']
            )

            # Track example usage
            example_used = None
            if predicted:
                example_used = predicted.get('example_used', '').lower() if predicted.get('example_used') else ''
                if example_used and example_used not in ['none', 'n/a', 'na', '']:
                    example_usage_stats['used'] += 1
                elif example_used in ['none', 'n/a', 'na']:
                    example_usage_stats['not_used'] += 1
                else:
                    example_usage_stats['unclear'] += 1

            results.append({
                'sample_id': idx,
                'query_image_id': query_image_id,
                'query_image_url': query_image_url,
                'query_source': query_source,
                'n_support': n_support,
                'support_strategy': sample.get('support_strategy', 'unknown'),
                'ground_truth': sample['query']['metadata'],
                'raw_response': response,
                'predicted': predicted,
                'evaluation': evaluation,
                # Few-shot specific tracking
                'example_used': predicted.get('example_used') if predicted else None,
                'example_reasoning': predicted.get('example_reasoning') if predicted else None,
            })

        except Exception as e:
            print(f"\nError at sample {idx}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'sample_id': idx,
                'query_image_id': sample['query']['image_id'],
                'error': str(e)
            })

    # Calculate statistics
    stats = calculate_statistics(results)

    # Add example usage statistics
    total_valid = example_usage_stats['used'] + example_usage_stats['not_used'] + example_usage_stats['unclear']
    stats['example_usage'] = {
        'examples_cited': example_usage_stats['used'],
        'no_example_cited': example_usage_stats['not_used'],
        'unclear_response': example_usage_stats['unclear'],
        'citation_rate': example_usage_stats['used'] / total_valid if total_valid > 0 else 0
    }

    # Save final results
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    model_name_short = tester.model_name.split('/')[-1]
    final_output = output_path / f"{model_name_short}_fewshot_{n_support}shot_{timestamp}.json"

    output_data = {
        'metadata': {
            'model': tester.model_name,
            'model_class': tester.__class__.__name__,
            'num_samples': len(dataset),
            'n_support': n_support,
            'prompt': 'fewshot_with_example_tracking',
            'timestamp': timestamp
        },
        'statistics': stats,
        'results': results
    }

    with open(final_output, 'w') as f:
        json.dump(output_data, f, indent=2)

    print(f"\nResults saved to: {final_output}")
    print_statistics_summary_fewshot(stats, n_support)

    return output_data

def print_statistics_summary_fewshot(stats: Dict, n_support: int):
    """Print statistics summary with example usage information."""
    print(f"\n{'='*60}")
    print(f"Evaluation Summary ({n_support}-Shot)")
    print('='*60)

    print(f"\nOutput Quality:")
    print(f"  Successful extractions: {stats['extraction_success_rate']:.1%}")

    print(f"\nGeographic Accuracy:")
    print(f"  Continent: {stats['continent_accuracy']:.1%}")
    print(f"  Country:   {stats['country_accuracy']:.1%}")
    print(f"  Region:    {stats['region_accuracy']:.1%}")
    print(f"  City:      {stats['city_accuracy']:.1%}")

    if stats.get('avg_distance_km') is not None:
        print(f"\nCoordinate Accuracy:")
        print(f"  Average distance:  {stats['avg_distance_km']:.1f} km")
        print(f"  Median distance:   {stats['median_distance_km']:.1f} km")
        print(f"  Within 1 km:       {stats['accuracy_1km']:.1%}")
        print(f"  Within 10 km:      {stats['accuracy_10km']:.1%}")
        print(f"  Within 100 km:     {stats['accuracy_100km']:.1%}")
        print(f"  Within 1000 km:    {stats['accuracy_1000km']:.1%}")

    # Example usage statistics
    if 'example_usage' in stats:
        eu = stats['example_usage']
        print(f"\nExample Utilization:")
        print(f"  Cited an example:    {eu['examples_cited']} ({eu['citation_rate']:.1%})")
        print(f"  No example cited:    {eu['no_example_cited']}")
        print(f"  Unclear/missing:     {eu['unclear_response']}")

    print('='*60)

### Prompts

Zero shot prompts

In [ ]:
xml_prompt_simple = """Analyze this indoor room image to determine its geographic location.

Provide your answer in this XML format:

<location>
  <continent>continent name</continent>
  <country>country name</country>
  <region>state/province/region name</region>
  <city>city name</city>
  <coordinates>
    <latitude>decimal number</latitude>
    <longitude>decimal number</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
  <property>
    <name>property/hotel name or "unknown"</name>
    <type>hotel/apartment/house or "unknown"</type>
  </property>
  <reasoning>Brief explanation of visual clues that led to this conclusion</reasoning>
</location>
"""

xml_prompt_detailed = """Analyze this indoor room image to determine its geographic location.

Based on visual clues such as architectural style, furniture, decor, electrical outlets,
window types, signage, and any other regional markers, determine where this room is located.

If you can identify any property branding, hotel chain logos, or distinctive design elements
that might help narrow down the location, include that information.

Provide your answer in this XML format:

<location>
  <continent>continent name</continent>
  <country>country name</country>
  <region>state/province/region name</region>
  <city>city name</city>
  <coordinates>
    <latitude>decimal number</latitude>
    <longitude>decimal number</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
  <property>
    <name>property/hotel name or "unknown"</name>
    <type>hotel/apartment/house or "unknown"</type>
  </property>
  <reasoning>Brief explanation of visual clues that led to this conclusion</reasoning>
</location>
"""

xml_prompt_analytical = """Analyze this indoor room image to determine its geographic location through systematic observation.

Step 1: Identify regional indicators (electrical outlets, wall sockets, light switches)
Step 2: Analyze architectural features (ceiling style, molding, window frames, door types)
Step 3: Examine furnishings (furniture style, bedding patterns, decor elements)
Step 4: Look for text/branding (signs, labels, property logos)
Step 5: Consider climate clues (HVAC type, window treatments, insulation indicators)
Step 6: Synthesize observations into location estimate

Provide your answer in this XML format:

<location>
  <continent>continent name</continent>
  <country>country name</country>
  <region>state/province/region name</region>
  <city>city name</city>
  <coordinates>
    <latitude>decimal number</latitude>
    <longitude>decimal number</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
  <property>
    <name>property/hotel name or "unknown"</name>
    <type>hotel/apartment/house or "unknown"</type>
  </property>
  <reasoning>Brief explanation of visual clues that led to this conclusion</reasoning>
</location>
"""

xml_prompt_attribution = """Analyze this room image to determine its location.
List the key visual features you observe and explain how each helps identify the location.

Provide output in XML format:

<location>
  <continent>continent name</continent>
  <country>country name</country>
  <region>region name</region>
  <city>city name</city>
  <coordinates>
    <latitude>decimal</latitude>
    <longitude>decimal</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
  <key_feature_1>Description of first important feature and why it indicates this location</key_feature_1>
  <key_feature_2>Description of second important feature and why it indicates this location</key_feature_2>
  <key_feature_3>Description of third important feature and why it indicates this location</key_feature_3>
  <reasoning>Overall synthesis of all evidence</reasoning>
</location>
"""

xml_prompt_hierarchical = """Analyze this room image to determine its location. Think step-by-step from continent to city.

Provide output in XML format:

<location>
  <continent>continent name</continent>
  <continent_reasoning>Explain why this continent</continent_reasoning>
  <country>country name</country>
  <country_reasoning>Explain what narrowed to this country</country_reasoning>
  <region>region name</region>
  <region_reasoning>Explain regional indicators</region_reasoning>
  <city>city name</city>
  <city_reasoning>Explain city-specific clues</city_reasoning>
  <coordinates>
    <latitude>decimal</latitude>
    <longitude>decimal</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
</location>
"""

Few shot prompt

In [ ]:
xml_prompt_fewshot = """You have seen example room images with their known locations.
Now analyze this query image to determine its geographic location.

Look for regional indicators like electrical outlets, architectural style, furniture, and signage.
If any example helped identify visual patterns, note which one.

Provide your answer in this XML format:

<location>
  <continent>continent name</continent>
  <country>country name</country>
  <region>state/province/region name</region>
  <city>city name</city>
  <coordinates>
    <latitude>decimal number</latitude>
    <longitude>decimal number</longitude>
  </coordinates>
  <confidence>high/medium/low</confidence>
  <example_used>none OR example number (e.g., "Example 2") that most helped</example_used>
  <example_reasoning>If an example helped, briefly explain what visual similarity you noticed. If none helped, write "No direct similarity found"</example_reasoning>
  <reasoning>Brief explanation of visual clues that led to this conclusion</reasoning>
</location>
"""

### Istantiate models

##### Gemma

In [ ]:
import torch
gemma_tester = GemmaGeolocationTester(
    model_name="google/gemma-3-4b-it",
    mode="pipeline",
    hotel50k_image_index="hotel50k/image_path_index.json",
    airbnb_image_index="airbnb/airbnb_image_lookup.pkl"
)

Hotel50k index: 857230 mappings loaded
Airbnb index: 1540616 mappings loaded
Initializing GemmaGeolocationTester with google/gemma-3-4b-it in pipeline mode
  Loading Gemma with optimizations...
Loading Gemma pipeline...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use cuda:0


  Pipeline ready


##### Qwen

In [ ]:
qwen_tester = QwenGeolocationTester(
    model_name="Qwen/Qwen2.5-VL-3B-Instruct",
    mode="model",  # Qwen only supports model mode
    hotel50k_image_index="hotel50k/image_path_index.json",
    airbnb_image_index="airbnb/airbnb_image_lookup.pkl"
)

Hotel50k index: 857230 mappings loaded
Airbnb index: 1540616 mappings loaded
Initializing QwenGeolocationTester with Qwen/Qwen2.5-VL-3B-Instruct in model mode
  Loading Qwen processor and model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model ready on cuda:0
VRAM: 7.6GB / 23.8GB


##### Llama

In [ ]:
# 1. Install dependencies
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Verify installation (This should print the path to ollama, usually /usr/local/bin/ollama)
!which ollama

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,297 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,868 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,611 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driver

In [ ]:
# Find the absolute path to the ollama executable
ollama_path = shutil.which("ollama")

if ollama_path is None:
    raise FileNotFoundError("Ollama binary not found. Please run the installation cell again.")

# 1. Start the server using the absolute path
ollama_process = subprocess.Popen(
    [ollama_path, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 2. Wait for the server to be "Live"
print("Waiting for Ollama server to start...")
for i in range(30):
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            print("Ollama is running!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("Timeout: Ollama server failed to start.")

# 3. Pull the vision model
!ollama pull llama3.2-vision:11b-instruct-q4_K_M

Waiting for Ollama server to start...
Ollama is running!



In [ ]:
ollama

<module 'ollama' from '/usr/local/lib/python3.12/dist-packages/ollama/__init__.py'>

In [ ]:
# Initialize Ollama tester with quantization handled by Ollama
ollama_tester = OllamaGeolocationTester(
    model_name="llama3.2-vision:11b-instruct-q4_K_M",  # Ollama model name format
    device="cuda",
    hotel50k_image_index="hotel50k/image_path_index.json",
    airbnb_image_index="airbnb/airbnb_image_lookup.pkl"
)

Hotel50k index: 857230 mappings loaded
Airbnb index: 1540616 mappings loaded
Initializing OllamaGeolocationTester with llama3.2-vision:11b-instruct-q4_K_M in auto mode
  Initializing Ollama client for llama3.2-vision:11b-instruct-q4_K_M...
Model llama3.2-vision:11b-instruct-q4_K_M is available
Ollama client ready


#####MiniCPM

In [ ]:
minicpm_tester = MiniCPMVGeolocationTesterOptimized(
    model_name="openbmb/MiniCPM-V-2_6",
    mode="model",
    hotel50k_image_index="hotel50k/image_path_index.json",
    airbnb_image_index="airbnb/airbnb_image_lookup.pkl"
)

Hotel50k index: 857230 mappings loaded
Airbnb index: 1540616 mappings loaded
Initializing MiniCPMVGeolocationTesterOptimized with openbmb/MiniCPM-V-2_6 in model mode
Loading MiniCPM-V-2.6 with speed optimizations...


config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

configuration_minicpm.py:   0%|          | 0.00/3.28k [00:00<?, ?B/s]

modeling_navit_siglip.py:   0%|          | 0.00/41.9k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.56k [00:00<?, ?B/s]

Verifying model device placement...
Model successfully loaded on cuda:0
Memory usage: 16.3GB / 23.8GB VRAM


#####LLaVA

In [ ]:
llava_tester = LLaVAGeolocationTester(
    model_name="llava-hf/llava-v1.6-mistral-7b-hf",
    mode="model",
    hotel50k_image_index="hotel50k/image_path_index.json",
    airbnb_image_index="airbnb/airbnb_image_lookup.pkl"
)

Hotel50k index: 857230 mappings loaded
Airbnb index: 1540616 mappings loaded
Initializing LLaVAGeolocationTester with llava-hf/llava-v1.6-mistral-7b-hf in model mode
Loading LLaVA-NeXT (4-bit quantized)...
  Using SDPA


preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

4-bit quantization + gradient checkpointing enabled
VRAM: 4.3GB / 23.8GB (saved ~7GB vs BF16)


###Zero shot

In [ ]:
prompts = ["simple", "detailed", "analytical", "attribution", "hierarchical"]
data_sources = ["hotel50k", "airbnb"]
dataset_types = ["Diverse", "Geographic"]

# Diverse is for few shot with support samples from different countries
# Geographic is for few shot with 2/3 of support samples from the same countries
# The zero shot json datasets have the same query target images

In [ ]:
dataset_type = dataset_types[0]
data_source = data_sources[0]
vlm_tester = llava_tester

zero_shot_str = "zero-shot-datasets-matched" if dataset_type == "Diverse" else "zero-shot-datasets-matched-mixed"

print(f"Dataset type: {dataset_type}")
print(f"Data source: {data_source}")

Dataset type: Diverse
Data source: hotel50k


#####Example Run

In [ ]:
vlm_tester = llava_tester
dataset_type = "Geographic"

zero_shot_str = "zero-shot-datasets-matched" if dataset_type == "Diverse" else "zero-shot-datasets-matched-mixed"

for data_source in data_sources:

    zero_shot_dataset_path = f"./{zero_shot_str}/{data_source}/{data_source}_zero_shot.json"
    zero_shot_output_dir = f"./experiments-results/zero-shot/{data_source}/{dataset_type}/"

    # verify everything is correct
    print(f"Dataset type {dataset_type}")
    print(f"Dataset path (zero shot): {zero_shot_dataset_path}")
    print(f"Output directory (zero shot): {zero_shot_output_dir}")

    for prompt_style in prompts:
        results_zero_shot = run_zeroshot_test(
            tester=vlm_tester,
            dataset_path=zero_shot_dataset_path,
            output_dir=zero_shot_output_dir,
            prompt_style=prompt_style,
            max_new_tokens=1024
        )

Dataset type Geographic
Dataset path (zero shot): ./zero-shot-datasets-matched-mixed/hotel50k/hotel50k_zero_shot.json
Output directory (zero shot): ./results/zero-shot/hotel50k/Geographic/
Loading dataset from ./zero-shot-datasets-matched-mixed/hotel50k/hotel50k_zero_shot.json...

Running zero-shot inference with 'simple' prompt on 100 samples...
Using individual processing...


Processing:   0%|          | 0/100 [00:00<?, ?it/s]/tmp/ipython-input-586466592.py:361: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  property_elem = root.find('property') or root.find('hotel')
Processing: 100%|██████████| 100/100 [53:46<00:00, 32.27s/it]



Results saved to: results/zero-shot/hotel50k/Geographic/llava-v1.6-mistral-7b-hf_zeroshot_simple_20260105_022111.json
Evaluation Summary (0-Shot)

Output Quality:
Successful extractions: 100.0%

Geographic Accuracy:
Continent: 77.0%
Country:   42.0%
Region:    12.0%
City:      1.0%

Coordinate Accuracy:
Average distance:  3652.2 km
Median distance:   1536.3 km
Within 1 km:       0.0%
Within 10 km:      2.0%
Within 100 km:     5.0%
Within 1000 km:    33.0%
Loading dataset from ./zero-shot-datasets-matched-mixed/hotel50k/hotel50k_zero_shot.json...

Running zero-shot inference with 'detailed' prompt on 100 samples...
Using individual processing...


Processing: 100%|██████████| 100/100 [47:03<00:00, 28.24s/it]



Results saved to: results/zero-shot/hotel50k/Geographic/llava-v1.6-mistral-7b-hf_zeroshot_detailed_20260105_030815.json
Evaluation Summary (0-Shot)

Output Quality:
Successful extractions: 100.0%

Geographic Accuracy:
Continent: 76.0%
Country:   41.0%
Region:    12.0%
City:      1.0%

Coordinate Accuracy:
Average distance:  3715.3 km
Median distance:   1477.9 km
Within 1 km:       0.0%
Within 10 km:      2.0%
Within 100 km:     5.0%
Within 1000 km:    33.0%
Loading dataset from ./zero-shot-datasets-matched-mixed/hotel50k/hotel50k_zero_shot.json...

Running zero-shot inference with 'analytical' prompt on 100 samples...
Using individual processing...


...

Processing:  73%|███████▎  | 73/100 [32:53<13:29, 29.97s/it]

Processing: 100%|██████████| 100/100 [44:37<00:00, 26.78s/it]



Results saved to: results/zero-shot/airbnb/Geographic/llava-v1.6-mistral-7b-hf_zeroshot_attribution_20260105_083541.json
Evaluation Summary (0-Shot)

Output Quality:
Successful extractions: 97.0%

Geographic Accuracy:
Continent: 61.0%
Country:   22.0%
Region:    4.0%
City:      1.0%

Coordinate Accuracy:
Average distance:  4736.2 km
Median distance:   2096.6 km
Within 1 km:       0.0%
Within 10 km:      2.0%
Within 100 km:     5.0%
Within 1000 km:    30.0%
Loading dataset from ./zero-shot-datasets-matched-mixed/airbnb/airbnb_zero_shot.json...

Running zero-shot inference with 'hierarchical' prompt on 100 samples...
Using individual processing...


Processing: 100%|██████████| 100/100 [37:44<00:00, 22.64s/it]


Results saved to: results/zero-shot/airbnb/Geographic/llava-v1.6-mistral-7b-hf_zeroshot_hierarchical_20260105_091325.json
Evaluation Summary (0-Shot)

Output Quality:
Successful extractions: 100.0%

Geographic Accuracy:
Continent: 71.0%
Country:   26.0%
Region:    4.0%
City:      1.0%

Coordinate Accuracy:
Average distance:  4222.1 km
Median distance:   1415.4 km
Within 1 km:       0.0%
Within 10 km:      5.0%
Within 100 km:     8.0%
Within 1000 km:    34.0%


In [ ]:
# Add at end of training cell
from google.colab import runtime
runtime.unassign()

###Few shot

#####Example Run

In [ ]:
# Check GPU memory before inference
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

VRAM used: 16.33 GB
VRAM reserved: 16.47 GB


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()

Few Shot Run

In [ ]:
data_sources = ["hotel50k", "airbnb"]
dataset_types = ["Diverse", "Geographic"]
n_shot = [5, 10, 15, 20]

In [ ]:
for data_source in data_sources:
    for dataset_type in dataset_types:
        for n in n_shot:
            few_shot_str = "fewshot-datasets" if dataset_type == "Diverse" else "fewshot-datasets-mixed"
            few_shot_dataset_path = f"./{few_shot_str}/{data_source}/New/fewshot_{n}shot.json"
            few_shot_output_dir = f"./experiments-results/few-shot/{data_source}/{dataset_type}/{n}shot"
            print(f"Output directory (few shot):{few_shot_output_dir}")
            results_few_shot = run_fewshot_test(
                tester=llava_tester,
                dataset_path=few_shot_dataset_path,
                output_dir=few_shot_output_dir,
                max_new_tokens=1024
            )

Output directory (few shot):./experiments-results/few-shot/hotel50k/Diverse/5shot
Loading few-shot dataset from ./fewshot-datasets/hotel50k/New/fewshot_5shot.json...
Loaded 100 samples with 5 support examples each

Running 5-shot inference...
Model: llava-hf/llava-v1.6-mistral-7b-hf


5-shot: 100%|██████████| 100/100 [50:51<00:00, 30.52s/it]



Results saved to: experiments-results/few-shot/hotel50k/Diverse/5shot/llava-v1.6-mistral-7b-hf_fewshot_5shot_20260117_165741.json

Evaluation Summary (5-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 47.0%
  Country:   17.0%
  Region:    3.0%
  City:      1.0%

Coordinate Accuracy:
  Average distance:  6247.2 km
  Median distance:   6724.8 km
  Within 1 km:       0.0%
  Within 10 km:      2.0%
  Within 100 km:     2.0%
  Within 1000 km:    18.0%

Example Utilization:
  Cited an example:    9 (9.0%)
  No example cited:    91
  Unclear/missing:     0
Output directory (few shot):./experiments-results/few-shot/hotel50k/Diverse/10shot
Loading few-shot dataset from ./fewshot-datasets/hotel50k/New/fewshot_10shot.json...
Loaded 100 samples with 10 support examples each

Running 10-shot inference...
Model: llava-hf/llava-v1.6-mistral-7b-hf


10-shot: 100%|██████████| 100/100 [59:08<00:00, 35.49s/it]



Results saved to: experiments-results/few-shot/hotel50k/Diverse/10shot/llava-v1.6-mistral-7b-hf_fewshot_10shot_20260117_175651.json

Evaluation Summary (10-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 29.0%
  Country:   3.0%
  Region:    1.0%
  City:      1.0%

Coordinate Accuracy:
  Average distance:  7270.1 km
  Median distance:   8125.0 km
  Within 1 km:       0.0%
  Within 10 km:      1.0%
  Within 100 km:     1.0%
  Within 1000 km:    14.0%

Example Utilization:
  Cited an example:    31 (31.0%)
  No example cited:    69
  Unclear/missing:     0
Output directory (few shot):./experiments-results/few-shot/hotel50k/Diverse/15shot
Loading few-shot dataset from ./fewshot-datasets/hotel50k/New/fewshot_15shot.json...
Loaded 100 samples with 15 support examples each

Running 15-shot inference...
Model: llava-hf/llava-v1.6-mistral-7b-hf


...


Error at sample 67: image file is truncated (51 bytes not processed)


20-shot:  93%|█████████▎| 93/100 [1:47:20<07:39, 65.58s/it]Traceback (most recent call last):
  File "/tmp/ipython-input-146648270.py", line 54, in run_fewshot_test
    response = tester.generate_prediction_fewshot(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1874673378.py", line 179, in generate_prediction_fewshot
    return self._run_inference(formatted_input, max_new_tokens)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1874673378.py", line 74, in _run_inference
    return self._run_fewshot(formatted_input, max_new_tokens)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1874673378.py", line 139, in _run_fewshot
    query_image = self._load_image_from_url(formatted_input['query_image_url'], force_url=True)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-3512097251.py", line 215, in _load_imag


Error at sample 93: 403 Client Error: Forbidden for url: https://a0.muscache.com/pictures/prohost-api/Hosting-1302858912250348888/original/2f2b11ed-22b7-4bd7-b927-f14c4a235399.jpeg


20-shot: 100%|██████████| 100/100 [1:54:56<00:00, 68.97s/it]


Results saved to: experiments-results/few-shot/airbnb/Geographic/20shot/llava-v1.6-mistral-7b-hf_fewshot_20shot_20260118_120238.json

Evaluation Summary (20-Shot)

Output Quality:
  Successful extractions: 94.0%

Geographic Accuracy:
  Continent: 52.0%
  Country:   32.0%
  Region:    9.0%
  City:      4.0%

Coordinate Accuracy:
  Average distance:  5364.2 km
  Median distance:   2148.8 km
  Within 1 km:       0.0%
  Within 10 km:      6.0%
  Within 100 km:     11.0%
  Within 1000 km:    38.0%

Example Utilization:
  Cited an example:    89 (94.7%)
  No example cited:    5
  Unclear/missing:     0


In [ ]:
from google.colab import runtime
runtime.unassign()

#####Temp (for llama)

In [ ]:
data_source = "hotel50k"
dataset_type = "Geographic"
n_shot = [5, 10, 15, 20]

for n in n_shot:
    few_shot_str = "fewshot-datasets" if dataset_type == "Diverse" else "fewshot-datasets-mixed"
    few_shot_dataset_path = f"./{few_shot_str}/{data_source}/New/fewshot_{n}shot.json"
    few_shot_output_dir = f"./experiments-results/few-shot/{data_source}/{dataset_type}/{n}shot"
    print(f"Dataset path (few shot): {few_shot_dataset_path}")
    print(f"Output directory (few shot):{few_shot_output_dir}")
    results_few_shot = run_fewshot_test(
        tester=ollama_tester,
        dataset_path=few_shot_dataset_path,
        output_dir=few_shot_output_dir,
        max_new_tokens=1024
    )

Dataset path (few shot): ./fewshot-datasets-mixed/hotel50k/New/fewshot_5shot.json
Output directory (few shot):./experiments-results/few-shot/hotel50k/Geographic/5shot
Loading few-shot dataset from ./fewshot-datasets-mixed/hotel50k/New/fewshot_5shot.json...
Loaded 100 samples with 5 support examples each

Running 5-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


5-shot: 100%|██████████| 100/100 [1:08:09<00:00, 40.90s/it]



Results saved to: experiments-results/few-shot/hotel50k/Geographic/5shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_5shot_20260116_024602.json

Evaluation Summary (5-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 43.0%
  Country:   36.0%
  Region:    8.0%
  City:      0.0%

Coordinate Accuracy:
  Average distance:  6846.1 km
  Median distance:   6878.7 km
  Within 1 km:       0.0%
  Within 10 km:      0.0%
  Within 100 km:     1.0%
  Within 1000 km:    3.0%

Example Utilization:
  Cited an example:    35 (35.0%)
  No example cited:    0
  Unclear/missing:     65
Dataset path (few shot): ./fewshot-datasets-mixed/hotel50k/New/fewshot_10shot.json
Output directory (few shot):./experiments-results/few-shot/hotel50k/Geographic/10shot
Loading few-shot dataset from ./fewshot-datasets-mixed/hotel50k/New/fewshot_10shot.json...
Loaded 100 samples with 10 support examples each

Running 10-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


10-shot: 100%|██████████| 100/100 [1:23:33<00:00, 50.14s/it]



Results saved to: experiments-results/few-shot/hotel50k/Geographic/10shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_10shot_20260116_040936.json

Evaluation Summary (10-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 49.0%
  Country:   43.0%
  Region:    12.0%
  City:      0.0%

Coordinate Accuracy:
  Average distance:  7788.4 km
  Median distance:   10450.4 km
  Within 1 km:       0.0%
  Within 10 km:      0.0%
  Within 100 km:     0.0%
  Within 1000 km:    1.0%

Example Utilization:
  Cited an example:    83 (83.0%)
  No example cited:    0
  Unclear/missing:     17
Dataset path (few shot): ./fewshot-datasets-mixed/hotel50k/New/fewshot_15shot.json
Output directory (few shot):./experiments-results/few-shot/hotel50k/Geographic/15shot
Loading few-shot dataset from ./fewshot-datasets-mixed/hotel50k/New/fewshot_15shot.json...
Loaded 100 samples with 15 support examples each

Running 15-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_

15-shot: 100%|██████████| 100/100 [1:50:43<00:00, 66.44s/it]



Results saved to: experiments-results/few-shot/hotel50k/Geographic/15shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_15shot_20260116_060021.json

Evaluation Summary (15-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 49.0%
  Country:   40.0%
  Region:    10.0%
  City:      2.0%

Example Utilization:
  Cited an example:    86 (86.0%)
  No example cited:    0
  Unclear/missing:     14
Dataset path (few shot): ./fewshot-datasets-mixed/hotel50k/New/fewshot_20shot.json
Output directory (few shot):./experiments-results/few-shot/hotel50k/Geographic/20shot
Loading few-shot dataset from ./fewshot-datasets-mixed/hotel50k/New/fewshot_20shot.json...
Loaded 100 samples with 20 support examples each

Running 20-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


20-shot: 100%|██████████| 100/100 [2:14:49<00:00, 80.90s/it]


Results saved to: experiments-results/few-shot/hotel50k/Geographic/20shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_20shot_20260116_081511.json

Evaluation Summary (20-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 43.0%
  Country:   39.0%
  Region:    12.0%
  City:      3.0%

Coordinate Accuracy:
  Average distance:  8463.9 km
  Median distance:   9472.0 km
  Within 1 km:       0.0%
  Within 10 km:      1.0%
  Within 100 km:     1.0%
  Within 1000 km:    1.0%

Example Utilization:
  Cited an example:    93 (93.0%)
  No example cited:    0
  Unclear/missing:     7


In [ ]:
data_source = "airbnb"
dataset_types = ["Diverse", "Geographic"]
n_shot = [5, 10, 15, 20]

for dataset_type in dataset_types:
    for n in n_shot:
        few_shot_str = "fewshot-datasets" if dataset_type == "Diverse" else "fewshot-datasets-mixed"
        few_shot_dataset_path = f"./{few_shot_str}/{data_source}/New/fewshot_{n}shot.json"
        few_shot_output_dir = f"./experiments-results/few-shot/{data_source}/{dataset_type}/{n}shot"
        print(f"Dataset path (few shot): {few_shot_dataset_path}")
        print(f"Output directory (few shot):{few_shot_output_dir}")
        results_few_shot = run_fewshot_test(
            tester=ollama_tester,
            dataset_path=few_shot_dataset_path,
            output_dir=few_shot_output_dir,
            max_new_tokens=1024
        )

Dataset path (few shot): ./fewshot-datasets/airbnb/New/fewshot_5shot.json
Output directory (few shot):./experiments-results/few-shot/airbnb/Diverse/5shot
Loading few-shot dataset from ./fewshot-datasets/airbnb/New/fewshot_5shot.json...
Loaded 100 samples with 5 support examples each

Running 5-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


5-shot: 100%|██████████| 100/100 [1:09:19<00:00, 41.60s/it]



Results saved to: experiments-results/few-shot/airbnb/Diverse/5shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_5shot_20260116_092432.json

Evaluation Summary (5-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 30.0%
  Country:   20.0%
  Region:    6.0%
  City:      1.0%

Coordinate Accuracy:
  Average distance:  5321.3 km
  Median distance:   4153.5 km
  Within 1 km:       0.0%
  Within 10 km:      0.0%
  Within 100 km:     0.0%
  Within 1000 km:    5.0%

Example Utilization:
  Cited an example:    81 (81.0%)
  No example cited:    0
  Unclear/missing:     19
Dataset path (few shot): ./fewshot-datasets/airbnb/New/fewshot_10shot.json
Output directory (few shot):./experiments-results/few-shot/airbnb/Diverse/10shot
Loading few-shot dataset from ./fewshot-datasets/airbnb/New/fewshot_10shot.json...
Loaded 100 samples with 10 support examples each

Running 10-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


10-shot: 100%|██████████| 100/100 [1:28:13<00:00, 52.94s/it]



Results saved to: experiments-results/few-shot/airbnb/Diverse/10shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_10shot_20260116_105246.json

Evaluation Summary (10-Shot)

Output Quality:
  Successful extractions: 100.0%

Geographic Accuracy:
  Continent: 28.0%
  Country:   18.0%
  Region:    3.0%
  City:      0.0%

Coordinate Accuracy:
  Average distance:  7799.8 km
  Median distance:   9616.3 km
  Within 1 km:       0.0%
  Within 10 km:      0.0%
  Within 100 km:     0.0%
  Within 1000 km:    2.0%

Example Utilization:
  Cited an example:    94 (94.0%)
  No example cited:    0
  Unclear/missing:     6
Dataset path (few shot): ./fewshot-datasets/airbnb/New/fewshot_15shot.json
Output directory (few shot):./experiments-results/few-shot/airbnb/Diverse/15shot
Loading few-shot dataset from ./fewshot-datasets/airbnb/New/fewshot_15shot.json...
Loaded 100 samples with 15 support examples each

Running 15-shot inference...
Model: llama3.2-vision:11b-instruct-q4_K_M


...

20-shot:  67%|██████▋   | 67/100 [1:35:09<38:29, 69.99s/it]

20-shot:  93%|█████████▎| 93/100 [2:14:27<10:08, 86.94s/it]Traceback (most recent call last):
  File "/tmp/ipython-input-3496346876.py", line 157, in _run_fewshot_inference
    query_image = self._load_image_from_url(query_image_url, force_url=True)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-3512097251.py", line 215, in _load_image_from_url
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 403 Client Error: Forbidden for url: https://a0.muscache.com/pictures/prohost-api/Hosting-1302858912250348888/original/2f2b11ed-22b7-4bd7-b927-f14c4a235399.jpeg

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipython-input-146648270.py", line 54, in run_fewshot_test
    response = tester.generate_prediction_few


Error at sample 93: Ollama few-shot failure: 403 Client Error: Forbidden for url: https://a0.muscache.com/pictures/prohost-api/Hosting-1302858912250348888/original/2f2b11ed-22b7-4bd7-b927-f14c4a235399.jpeg


20-shot: 100%|██████████| 100/100 [2:23:37<00:00, 86.17s/it]


Results saved to: experiments-results/few-shot/airbnb/Geographic/20shot/llama3.2-vision:11b-instruct-q4_K_M_fewshot_20shot_20260116_215310.json

Evaluation Summary (20-Shot)

Output Quality:
  Successful extractions: 95.0%

Geographic Accuracy:
  Continent: 41.0%
  Country:   21.0%
  Region:    5.0%
  City:      1.0%

Coordinate Accuracy:
  Average distance:  3643.9 km
  Median distance:   1205.0 km
  Within 1 km:       0.0%
  Within 10 km:      0.0%
  Within 100 km:     1.0%
  Within 1000 km:    6.0%

Example Utilization:
  Cited an example:    84 (88.4%)
  No example cited:    0
  Unclear/missing:     11


In [ ]:
from google.colab import runtime
runtime.unassign()